In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt
import re

In [2]:
# Load swing data
file_path = r"C:\Users\brend\OneDrive - Stonehill College\pybaseball_local\Updated_Swing_Data.csv"
df = pd.read_csv(file_path)

# Load bio data for height/weight
bio_path = r"C:\Users\brend\Downloads\biodata\biofile0.csv"
bio = pd.read_csv(bio_path)

In [3]:
def clean_name(name):
    if pd.isna(name):
        return name

    name = name.replace("’", "'")
    name = re.sub(r'\b(Jr|Sr|II|III|IV)\.?$', '', name)
    name = name.replace(".", "")
    name = re.sub(r'\s+', ' ', name)
    name = name.lower().strip()

    nicknames = {
        "cj": "c j",
        "geraldo": "gerardo",
    }

    for k, v in nicknames.items():
        name = re.sub(r'\b{}\b'.format(k), v, name)

    return name


bio['name'] = bio['usename'] + " " + bio['lastname']
bio['name_clean'] = (bio['usename'] + " " + bio['lastname']).apply(clean_name)

df['name_clean'] = df['name'].apply(clean_name)

name_fixes = {
    "cj abrams": "carl abrams",
    "joshua palacios": "josh palacios",
    "yuli gurriel": "yulieski gurriel",
    "gio urshela": "giovanny urshela",
    "michael siani": "mike siani"
}

df['name_clean'] = df['name_clean'].replace(name_fixes)

bio = bio[['name_clean','height','weight']]
df = df.merge(bio, on='name_clean', how='left')

In [4]:
df['side_enc'] = df['side'].map({'L':0,'R':1})

df.rename(columns={
    'avg_bat_speed': 'bat_speed',
    'swing_tilt': 'vert_tilt',
    'avg_swing_length': 'swing_len',
    'avg_intercept_y_vs_batter': 'contact_y'
}, inplace=True)

df['contact_events'] = df['Contact%'] * df['competitive_swings']
df['miss_events'] = df['competitive_swings'] - df['contact_events']

In [5]:
def run_cswing(df):

    def make_features(df):

        df = df.copy()

        numeric_cols = [
            'bat_speed','vert_tilt','attack_angle','swing_len',
            'contact_y','height','weight','attack_direction'
        ]

        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        eps = 1e-3

        df['bat_speed_sq'] = df['bat_speed']**2
        df['vert_tilt_sq'] = df['vert_tilt']**2
        df['attack_angle_sq'] = df['attack_angle']**2
        df['swing_len_sq'] = df['swing_len']**2

        df['bat_speed_cu'] = df['bat_speed']**3
        df['vert_tilt_cu'] = df['vert_tilt']**3
        df['attack_angle_cu'] = df['attack_angle']**3

        df['bat_speed_swing_len'] = df['bat_speed'] * df['swing_len']
        df['bat_speed_vert_tilt'] = df['bat_speed'] * df['vert_tilt']
        df['vert_tilt_swing_len'] = df['vert_tilt'] * df['swing_len']

        df['attack_angle_swing_len'] = df['attack_angle'] * df['swing_len']
        df['attack_angle_vert_tilt'] = df['attack_angle'] * df['vert_tilt']
        df['attack_angle_contact'] = df['attack_angle'] * df['contact_y']

        df['attack_rad'] = np.deg2rad(df['attack_angle'])
        df['attack_sin'] = np.sin(df['attack_rad'])
        df['attack_cos'] = np.cos(df['attack_rad'])
        df['attack_sin2'] = np.sin(2 * df['attack_rad'])
        df['attack_cos2'] = np.cos(2 * df['attack_rad'])

        df['tilt_len_ratio'] = df['vert_tilt'] / df['swing_len'].clip(lower=eps)
        df['tilt_angle_ratio'] = df['vert_tilt'] / df['attack_angle'].clip(lower=eps)

        df['tilt_len_ratio_log'] = np.log(df['tilt_len_ratio'] + eps)
        df['tilt_angle_ratio_log'] = np.log(np.abs(df['tilt_angle_ratio']) + eps)

        df['tilt_angle_int'] = df['vert_tilt'] * df['attack_angle']
        df['vert_reach'] = df['attack_angle'] - df['contact_y']

        df['BMI'] = df['weight'] / (df['height']**2) * 703
        df['height_weight'] = df['height'] * df['weight']

        return df


    df = make_features(df)

    feature_cols = [f for f in df.columns if f in [
        'bat_speed','vert_tilt','attack_angle','swing_len','contact_y',
        'side_enc','attack_direction',
        'tilt_angle_int','vert_reach',
        'bat_speed_sq','vert_tilt_sq','attack_angle_sq','swing_len_sq',
        'bat_speed_cu','vert_tilt_cu','attack_angle_cu',
        'tilt_len_ratio_log','tilt_angle_ratio_log',
        'attack_sin','attack_cos','attack_sin2','attack_cos2',
        'bat_speed_swing_len','bat_speed_vert_tilt','vert_tilt_swing_len',
        'attack_angle_swing_len','attack_angle_vert_tilt',
        'attack_angle_contact',
        'BMI','height_weight'
    ] if f in df.columns]

    skew = ['bat_speed','swing_len','attack_angle']
    pt = PowerTransformer(method='yeo-johnson')
    df[skew] = pt.fit_transform(df[skew])

    scaler = RobustScaler()
    df[['BMI']] = scaler.fit_transform(df[['BMI']])

    df['contact_rate'] = df['Contact%'].clip(0.001, 0.999)

    X = df[feature_cols]
    y = df['contact_rate']

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    preds = np.zeros(len(df))
    shap_vals = []

    for train_idx, test_idx in kf.split(X):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        weights = df.iloc[train_idx]['competitive_swings']

        model = XGBRegressor(
            objective='reg:logistic',
            n_estimators=3000,
            learning_rate=0.02,
            max_depth=6,
            min_child_weight=1,
            gamma=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=1.0,
            tree_method='hist',
            random_state=42,
            early_stopping_rounds=50
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=weights,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        fold_preds = model.predict(X_test)
        fold_preds = np.clip(fold_preds, 0.001, 0.999)

        preds[test_idx] = fold_preds

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)

        fold_shap = pd.Series(
            np.abs(shap_values).mean(axis=0),
            index=X.columns
        )

        shap_vals.append(fold_shap)

    shap_df = pd.concat(shap_vals, axis=1)
    shap_df.columns = [f'fold_{i+1}' for i in range(shap_df.shape[1])]
    shap_df['mean_imp'] = shap_df.mean(axis=1)

    thresh = 0.005
    shap_df['stable'] = (shap_df.iloc[:, :5] > thresh).mean(axis=1)

    shap_df = shap_df.sort_values('mean_imp', ascending=False)

    important = shap_df[shap_df['stable'] >= 0.6].index.tolist()

    X_final = df[important]
    y_final = y

    final_model = XGBRegressor(
        objective='reg:logistic',
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=6,
        min_child_weight=1,
        gamma=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1.0,
        tree_method='hist',
        random_state=42
    )

    final_model.fit(
        X_final,
        y_final,
        sample_weight=df['competitive_swings']
    )

    pred_contact = preds
    true_contact = df['Contact%']

    rmse = np.sqrt(mean_squared_error(true_contact, pred_contact))
    r2 = r2_score(true_contact, pred_contact)

    print("Cswing+ RMSE:", round(rmse, 4))
    print("Cswing+ R2:", round(r2, 4))

    df['pContact_model_oof'] = preds

    side_params = {}

    full_preds = final_model.predict(X_final)
    
    for side in [0,1]:
        mask = df['side_enc'] == side

        mean_pred = full_preds[mask].mean()
        std_pred = full_preds[mask].std()

        side_params[side] = {"mean":mean_pred,"std":std_pred}

        df.loc[mask,'cSwing+'] = 100 + 10 * (full_preds[mask] - mean_pred) / std_pred

    return df, final_model, important, make_features, side_params, pt, scaler

In [ ]:
def run_dswing(df):

    def make_features(df):

        eps = 1e-3
        
        # Swing Features
        df['bat_speed_sq'] = df['bat_speed']**2
        df['vert_tilt_sq'] = df['vert_tilt']**2
        df['attack_angle_sq'] = df['attack_angle']**2
        df['swing_len_sq'] = df['swing_len']**2
        
        df['bat_speed_cu'] = df['bat_speed']**3
        df['vert_tilt_cu'] = df['vert_tilt']**3
        df['attack_angle_cu'] = df['attack_angle']**3

        df['bat_speed_swing_len'] = df['bat_speed'] * df['swing_len']
        df['bat_speed_vert_tilt'] = df['bat_speed'] * df['vert_tilt']
        df['vert_tilt_swing_len'] = df['vert_tilt'] * df['swing_len']
        df['attack_angle_swing_len'] = df['attack_angle'] * df['swing_len']
        df['attack_angle_vert_tilt'] = df['attack_angle'] * df['vert_tilt']
        df['attack_angle_contact'] = df['attack_angle'] * df['contact_y']
        
        df['attack_rad'] = np.deg2rad(df['attack_angle'])
        df['attack_sin'] = np.sin(df['attack_rad'])
        df['attack_cos'] = np.cos(df['attack_rad'])
        df['attack_sin2'] = np.sin(2 * df['attack_rad'])
        df['attack_cos2'] = np.cos(2 * df['attack_rad'])
        
        df['tilt_len_ratio'] = df['vert_tilt'] / df['swing_len'].clip(lower=eps)
        df['tilt_angle_ratio'] = df['vert_tilt'] / df['attack_angle'].clip(lower=eps)
        df['tilt_len_ratio_log'] = np.log(df['tilt_len_ratio'] + eps)
        df['tilt_angle_ratio_log'] = np.log(np.abs(df['tilt_angle_ratio']) + eps)
        
        df['tilt_angle_int'] = df['vert_tilt'] * df['attack_angle']
        df['vert_reach'] = df['attack_angle'] - df['contact_y']
        df['bat_speed_tilt'] = df['bat_speed'] * df['vert_tilt']
        df['bat_speed_attack'] = df['bat_speed'] * df['attack_angle']
        df['tilt_swing_len'] = df['vert_tilt'] * df['swing_len']
        df['attack_angle_contact'] = df['attack_angle'] * df['contact_y']
        
        df['BMI'] = df['weight'] / (df['height']**2) * 703
        df['height_weight'] = df['height'] * df['weight']

        df['attack_dir_rad'] = np.deg2rad(df['attack_direction'])

        df['pull_bat_speed'] = df['bat_speed'] * np.cos(df['attack_dir_rad'])
        df['oppo_bat_speed'] = df['bat_speed'] * np.sin(df['attack_dir_rad'])

        df['pull_bat_speed_sq'] = df['pull_bat_speed'] ** 2

        df['bat_speed_attack_dir'] = df['bat_speed'] * df['attack_direction']

        df['pull_launch_eff'] = df['attack_angle'] * np.cos(df['attack_dir_rad'])

        df['effective_bat_speed'] = df['bat_speed'] * np.cos(
            np.deg2rad(df['attack_angle'] - df['vert_tilt'])
        )

        df['plane_align'] = np.abs(df['attack_angle'] - df['vert_tilt'])
        df['plane_align_inv'] = 1 / (df['plane_align'] + 1)

        df['dir_barrel_speed'] = (
            df['bat_speed']
            * np.cos(df['attack_dir_rad'])
            * np.cos(np.deg2rad(df['attack_angle'] - df['vert_tilt']))
        )

        df['pull_contact_power'] = df['pull_bat_speed'] * df['contact_y']

        df['contact_quality'] = (
            df['pull_bat_speed']
            * np.exp(-abs(df['attack_angle'] - df['vert_tilt']) / 10)
        )

        df['barrel_power'] = (
            df['bat_speed']
            * np.cos(df['attack_dir_rad'])
            * np.cos(np.deg2rad(df['attack_angle'] - df['vert_tilt']))
        )

        df["bat_speed_per_height"] = df["bat_speed"] / df["height"]
        df["swing_len_per_height"] = df["swing_len"] / df["height"]
        
        return df


    df = make_features(df)

    feature_cols = [f for f in df.columns if f in [

        'bat_speed','vert_tilt','attack_angle','swing_len','contact_y',
        'side_enc','attack_direction',

        'tilt_angle_int','vert_reach','bat_speed_tilt','bat_speed_attack',
        'tilt_swing_len','attack_angle_contact',

        'bat_speed_sq','vert_tilt_sq','attack_angle_sq','swing_len_sq',
        'bat_speed_cu','vert_tilt_cu','attack_angle_cu',

        'tilt_len_ratio_log','tilt_angle_ratio_log',

        'attack_sin','attack_cos','attack_sin2','attack_cos2',

        'attack_angle_swing_len','attack_angle_vert_tilt',
        'attack_angle_contact','bat_speed_swing_len',
        'bat_speed_vert_tilt','vert_tilt_swing_len',

        'BMI','height_weight',

        'pull_bat_speed',
        'oppo_bat_speed',
        'pull_bat_speed_sq',
        'bat_speed_attack_dir',
        'pull_launch_eff',
        'effective_bat_speed',
        'plane_align',
        'plane_align_inv',
        'dir_barrel_speed',
        'pull_contact_power',
        'contact_quality',
        'barrel_power',
        'bat_speed_per_height',
        'swing_len_per_height'

    ] if f in df.columns]

    skew = ['bat_speed','swing_len','attack_angle']
    pt = PowerTransformer(method='yeo-johnson')
    df[skew] = pt.fit_transform(df[skew])

    scaler = RobustScaler()
    df[['BMI']] = scaler.fit_transform(df[['BMI']])

    y = df['xwobacon']
    X = df[feature_cols]

    gkf = GroupKFold(n_splits=5)

    preds = np.zeros(len(df))
    shap_interactions_list = []

    for train_idx, test_idx in gkf.split(X, y, groups=df['name']):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        weights = df.iloc[train_idx]['competitive_swings']

        model = XGBRegressor(
            objective='reg:logistic',
            n_estimators=3000,
            learning_rate=0.02,
            max_depth=4,
            min_child_weight=1,
            gamma=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=1.0,
            tree_method='hist',
            random_state=42,
            early_stopping_rounds=50
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=weights,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        preds[test_idx] = model.predict(X_test)

        explainer = shap.TreeExplainer(model)
        shap_interactions = explainer.shap_interaction_values(X_test)
        shap_interactions_list.append(shap_interactions.mean(axis=0))

    mean_shap = np.mean(np.abs(np.array(shap_interactions_list)), axis=0)
    feature_mean_importance = mean_shap.sum(axis=1)

    important_features = [f for f, s in zip(feature_cols, feature_mean_importance) if s > 0.001]

    X_final = df[important_features]
    y_final = y

    final_model = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=1,
        gamma=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1.0,
        tree_method='hist',
        random_state=42
    )

    final_model.fit(
        X_final,
        y_final,
        sample_weight=df['competitive_swings']
    )

    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)

    print("dSwing+ RMSE:", round(rmse, 4))
    print("dSwing+ R2:", round(r2, 4))

    df['pxwobacon_model_oof'] = preds

    side_params = {}

    full_preds = final_model.predict(X_final)

    for side in [0,1]:

        mask = df['side_enc'] == side

        mean_pred = full_preds[mask].mean()
        std_pred = full_preds[mask].std()

        side_params[side] = {"mean":mean_pred,"std":std_pred}

        df.loc[mask,'dSwing+'] = 100 + 10 * (full_preds[mask] - mean_pred) / std_pred

    return df, final_model, important_features, make_features, side_params, pt, scaler

In [7]:
df_cswing, cs_model, feature_cols_cs, cs_create_features, cs_side_params, cs_pt, cs_scaler = run_cswing(df.copy())

df_dswing, ds_model, feature_cols_ds, ds_create_features, ds_side_params, ds_pt, ds_scaler = run_dswing(df.copy())

Cswing+ RMSE: 0.0463
Cswing+ R2: 0.5001
dSwing+ RMSE: 0.0378
dSwing+ R2: 0.5416


In [8]:
combined = df_cswing[['year','name','cSwing+']].merge(
    df_dswing[['year','name','dSwing+']],
    on=['year','name'],
    how='outer'
)

combined.to_excel(
    r"C:\Users\brend\OneDrive - Stonehill College\swing_plus_combined.xlsx",
    index=False
)

In [15]:
# Lookup tool for a player's Swing+ values
year_input = input("Enter the player's year: ")
name_input = input("Enter the player's full name: ")

name_cleaned = clean_name(name_input)

mask = (combined['year'] == int(year_input)) & (combined['name'].str.lower() == name_input.lower())

if mask.any():
    row = combined[mask].iloc[0]

    print(f"\n{row['name']} ({row['year']})")
    print(f"cSwing+: {row['cSwing+']}")
    print(f"dSwing+: {row['dSwing+']}")
else:
    print("Player not found.")


Cole Young (2025)
cSwing+: 100.47789764404297
dSwing+: 90.24210357666016


In [10]:
def simulate_swing_change(player_name, year, tweaks):

    player_row = df[(df['name'] == player_name) & (df['year'] == year)]

    if player_row.empty:
        print("Player not found.")
        return

    base = player_row.iloc[0].copy()
    sim = base.copy()

    # Apply swing tweaks
    for feat, change in tweaks.items():
        sim[feat] += change

    base_df = pd.DataFrame([base])
    sim_df = pd.DataFrame([sim])

    base_cs = cs_create_features(base_df)
    sim_cs = cs_create_features(sim_df)

    skew = ['bat_speed','swing_len','attack_angle']

    base_cs[skew] = cs_pt.transform(base_cs[skew])
    sim_cs[skew] = cs_pt.transform(sim_cs[skew])

    base_cs[['BMI']] = cs_scaler.transform(base_cs[['BMI']])
    sim_cs[['BMI']] = cs_scaler.transform(sim_cs[['BMI']])

    base_cs = base_cs[feature_cols_cs]
    sim_cs = sim_cs[feature_cols_cs]

    base_ds = ds_create_features(base_df)
    sim_ds = ds_create_features(sim_df)

    base_ds[skew] = ds_pt.transform(base_ds[skew])
    sim_ds[skew] = ds_pt.transform(sim_ds[skew])

    base_ds[['BMI']] = ds_scaler.transform(base_ds[['BMI']])
    sim_ds[['BMI']] = ds_scaler.transform(sim_ds[['BMI']])

    base_ds = base_ds[feature_cols_ds]
    sim_ds = sim_ds[feature_cols_ds]

    base_contact = cs_model.predict(base_cs)[0]
    sim_contact = cs_model.predict(sim_cs)[0]

    base_damage = ds_model.predict(base_ds)[0]
    sim_damage = ds_model.predict(sim_ds)[0]

    side = int(base["side_enc"])

    base_cswing = 100 + 10 * (base_contact - cs_side_params[side]["mean"]) / cs_side_params[side]["std"]
    sim_cswing = 100 + 10 * (sim_contact - cs_side_params[side]["mean"]) / cs_side_params[side]["std"]

    base_dswing = 100 + 10 * (base_damage - ds_side_params[side]["mean"]) / ds_side_params[side]["std"]
    sim_dswing = 100 + 10 * (sim_damage - ds_side_params[side]["mean"]) / ds_side_params[side]["std"]

    print(f"\nSimulation for {player_name} ({year})")

    print("\nTweaks:")
    for k,v in tweaks.items():
        print(f"{k}: {v:+}")

    print("\nRaw Predicted Values:")
    print(f"Contact% Before: {base_contact:.4f}")
    print(f"Contact% After : {sim_contact:.4f}")
    print(f"Change        : {sim_contact - base_contact:+.4f}")

    print(f"xwOBAcon Before: {base_damage:.4f}")
    print(f"xwOBAcon After : {sim_damage:.4f}")
    print(f"Change         : {sim_damage - base_damage:+.4f}")

    print("\nSwing+ Results:")

    print(f"\ncSwing+")
    print(f"Before: {base_cswing:.2f}")
    print(f"After : {sim_cswing:.2f}")
    print(f"Change: {sim_cswing - base_cswing:+.2f}")

    print(f"\ndSwing+")
    print(f"Before: {base_dswing:.2f}")
    print(f"After : {sim_dswing:.2f}")
    print(f"Change: {sim_dswing - base_dswing:+.2f}")

In [16]:
simulate_swing_change(
    "Cole Young",
    2025,
    {
        "bat_speed": 1,
    }
)


Simulation for Cole Young (2025)

Tweaks:
bat_speed: +1

Raw Predicted Values:
Contact% Before: 0.7554
Contact% After : 0.7666
Change        : +0.0112
xwOBAcon Before: 0.3197
xwOBAcon After : 0.3317
Change         : +0.0120

Swing+ Results:

cSwing+
Before: 100.48
After : 102.25
Change: +1.77

dSwing+
Before: 90.24
After : 92.50
Change: +2.26
